<a href="https://colab.research.google.com/github/razzz27/data-analysis-project/blob/main/Raj_Ranjan_2026B115410237.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hospital Appointment Operations Analysis
**Project 9 | Beginner-friendly Python Data Analysis | First-year B.Tech**

### Student Details
* **Name:** Raj Ranjan
* **Roll/Student No:** 2026B115410237
* **Branch:** CS-DS 2

## Project objective
Analyze appointment attendance, waiting times, department workload, SMS reminders, appointment types, and consultation fees. This is an **operational dataset**, not a clinical dataset. The notebook uses calculations and text/table outputs only—**no graphs or charts**.

## 1. Import modules and load the dataset
Run this notebook in Google Colab. The upload cell asks you to select `09_hospital_appointments.csv`.

In [44]:
import pandas as pd
import numpy as np
import os
from google.colab import files

# Fallback mechanism: Check if the file already exists locally to avoid files.upload() crashes on re-runs
file_name = "09_hospital_appointments.csv"

if os.path.exists(file_name):
    print(f"Found '{file_name}' in local directory. Loading directly...")
    df = pd.read_csv(file_name)
else:
    print(f"'{file_name}' not found locally. Please upload it:")
    try:
        uploaded = files.upload()
        if uploaded:
            file_name = next(iter(uploaded))
            df = pd.read_csv(file_name)
        else:
            raise FileNotFoundError("No file was uploaded.")
    except Exception as e:
        print(f"Upload error or cancelled: {e}")
        print("Looking for any alternative CSV files uploaded...")
        csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
        if csv_files:
            file_name = csv_files[0]
            print(f"Loading fallback file: {file_name}")
            df = pd.read_csv(file_name)
        else:
            raise FileNotFoundError("Please upload '09_hospital_appointments.csv' manually in the left sidebar files tab.")

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

Found '09_hospital_appointments.csv' in local directory. Loading directly...
Dataset loaded successfully.
Rows: 500
Columns: 9

Column names:
['Appointment_ID', 'Age', 'Department', 'Appointment_Date', 'Wait_Time_Min', 'Appointment_Type', 'SMS_Reminder', 'Attendance_Status', 'Consultation_Fee']


## 2. Inspect the appointment records

In [25]:
print("First five records:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nNumeric summary:")
display(df.describe())

First five records:


,Appointment_ID,Age,Department,Appointment_Date,Wait_Time_Min,Appointment_Type,SMS_Reminder,Attendance_Status,Consultation_Fee
0,AP00000,18,General,2025-05-04,37.0,Follow-up,Yes,Attended,1200
1,AP00001,32,Pediatrics,2025-01-22,31.0,Follow-up,Yes,Attended,1200
2,AP00002,46,General,2025-02-26,33.0,New,Yes,Attended,700
3,AP00003,49,Cardiology,2025-05-03,11.0,New,No,Attended,1200
4,AP00004,47,Cardiology,2025-05-05,26.0,Follow-up,No,Attended,1200



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Appointment_ID     500 non-null    object 
 1   Age                500 non-null    int64  
 2   Department         500 non-null    object 
 3   Appointment_Date   500 non-null    object 
 4   Wait_Time_Min      490 non-null    float64
 5   Appointment_Type   500 non-null    object 
 6   SMS_Reminder       500 non-null    object 
 7   Attendance_Status  500 non-null    object 
 8   Consultation_Fee   500 non-null    int64  
dtypes: float64(1), int64(2), object(6)
memory usage: 35.3+ KB

Numeric summary:


,Age,Wait_Time_Min,Consultation_Fee
count,500.000000,490.000000,500.000000
mean,43.066000,28.157143,731.600000
std,25.518072,14.433325,310.215905
min,1.000000,2.000000,300.000000
25%,21.000000,18.000000,500.000000
50%,41.500000,28.000000,700.000000
75%,65.000000,38.000000,900.000000
max,89.000000,71.000000,1200.000000


## 3. Variables, data types and basic validation
The dataset contains appointment ID, age, department, date, waiting time, appointment type, SMS reminder, attendance status and consultation fee.

In [26]:
required_columns = [
    "Appointment_ID", "Age", "Department", "Appointment_Date",
    "Wait_Time_Min", "Appointment_Type", "SMS_Reminder",
    "Attendance_Status", "Consultation_Fee"
]

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError("Required columns are missing: " + str(missing_columns))

print("All required columns are available.")
print("\nData types:")
print(df.dtypes)

All required columns are available.

Data types:
Appointment_ID        object
Age                    int64
Department            object
Appointment_Date      object
Wait_Time_Min        float64
Appointment_Type      object
SMS_Reminder          object
Attendance_Status     object
Consultation_Fee       int64
dtype: object


## 4. Data cleaning and transformation
- Convert appointment dates into date values.
- Convert waiting time to numeric values.
- Identify missing waiting times and duplicate appointment IDs.
- Keep the original missing-value count visible, then create a cleaned working DataFrame.
- Missing waiting times are filled with the overall median waiting time for analysis.

In [27]:
data = df.copy()

data["Appointment_Date"] = pd.to_datetime(data["Appointment_Date"], errors="coerce")
data["Wait_Time_Min"] = pd.to_numeric(data["Wait_Time_Min"], errors="coerce")
data["Consultation_Fee"] = pd.to_numeric(data["Consultation_Fee"], errors="coerce")

print("Missing values before cleaning:")
print(data.isnull().sum())

print("\nDuplicate Appointment_ID values:")
print(data["Appointment_ID"].duplicated().sum())

# Remove duplicate appointment IDs, keeping the first record.
data = data.drop_duplicates(subset="Appointment_ID", keep="first").copy()

# Fill missing waiting times using the median.
median_wait = data["Wait_Time_Min"].median()
data["Wait_Time_Min"] = data["Wait_Time_Min"].fillna(median_wait)

# Remove records with invalid dates or missing essential category/fee fields.
data = data.dropna(subset=[
    "Appointment_Date", "Department", "Appointment_Type",
    "SMS_Reminder", "Attendance_Status", "Consultation_Fee"
]).copy()

print("\nRows after cleaning:", len(data))
print("Missing waiting times after cleaning:", data["Wait_Time_Min"].isna().sum())
print("Median used to fill missing waiting times:", median_wait)

Missing values before cleaning:
Appointment_ID        0
Age                   0
Department            0
Appointment_Date      0
Wait_Time_Min        10
Appointment_Type      0
SMS_Reminder          0
Attendance_Status     0
Consultation_Fee      0
dtype: int64

Duplicate Appointment_ID values:
0

Rows after cleaning: 500
Missing waiting times after cleaning: 0
Median used to fill missing waiting times: 28.0


## 5. Create useful variables and categories

In [28]:
# Add a month column for date-wise summaries.
data["Month"] = data["Appointment_Date"].dt.to_period("M").astype(str)

# Categorize waiting times using conditional logic.
def waiting_category(minutes):
    if minutes <= 15:
        return "Short (0-15 min)"
    elif minutes <= 30:
        return "Moderate (16-30 min)"
    elif minutes <= 45:
        return "Long (31-45 min)"
    else:
        return "Very long (over 45 min)"

data["Waiting_Time_Category"] = data["Wait_Time_Min"].apply(waiting_category)

print("Waiting-time category counts:")
print(data["Waiting_Time_Category"].value_counts().to_string())

Waiting-time category counts:
Waiting_Time_Category
Moderate (16-30 min)       199
Long (31-45 min)           145
Short (0-15 min)            95
Very long (over 45 min)     61


## 6. Total appointments and attendance percentages
Attendance percentages use the total cleaned appointment count as the denominator. The percentage for each status is calculated as:

`status count / total appointments * 100`

In [29]:
total_appointments = len(data)
attendance_counts = data["Attendance_Status"].value_counts()
attendance_percentages = (attendance_counts / total_appointments) * 100

print("Total appointments:", total_appointments)
print("\nAttendance status counts:")
print(attendance_counts.to_string())
print("\nAttendance status percentages:")
for status, percentage in attendance_percentages.items():
    print(f"{status}: {percentage:.2f}%")

Total appointments: 500

Attendance status counts:
Attendance_Status
Attended     397
No-show       60
Cancelled     43

Attendance status percentages:
Attended: 79.40%
No-show: 12.00%
Cancelled: 8.60%


## 7. Department summary and average waiting time

In [30]:
department_summary = data.groupby("Department").agg(
    Total_Appointments=("Appointment_ID", "count"),
    Average_Wait_Min=("Wait_Time_Min", "mean"),
    Average_Consultation_Fee=("Consultation_Fee", "mean")
).round(2).sort_values("Total_Appointments", ascending=False)

print("Department summary:")
display(department_summary)

print("\nDepartment with the highest appointment volume:")
print(department_summary["Total_Appointments"].idxmax())

Department summary:


,Total_Appointments,Average_Wait_Min,Average_Consultation_Fee
Department,,,
Pediatrics,107,30.38,748.60
Dermatology,105,29.87,748.57
Cardiology,98,25.84,688.78
General,98,27.14,777.55
Orthopedics,92,27.15,689.13



Department with the highest appointment volume:
Pediatrics


## 8. No-show percentages by department

In [31]:
# Treat the Attendance_Status value 'No-show' as the no-show category.
no_show_rows = data[data["Attendance_Status"].astype(str).str.strip().str.lower() == "no-show"]

department_no_show = data.groupby("Department").agg(
    Total_Appointments=("Appointment_ID", "count")
)
department_no_show["No_Show_Count"] = no_show_rows.groupby("Department")["Appointment_ID"].count()
department_no_show["No_Show_Count"] = department_no_show["No_Show_Count"].fillna(0).astype(int)
department_no_show["No_Show_Percentage"] = (
    department_no_show["No_Show_Count"] / department_no_show["Total_Appointments"] * 100
).round(2)

print("No-show analysis by department:")
display(department_no_show.sort_values("No_Show_Percentage", ascending=False))

if len(no_show_rows) > 0:
    print("\nDepartment with the highest no-show percentage:",
          department_no_show["No_Show_Percentage"].idxmax())
else:
    print("\nNo 'No-show' attendance records were found in this dataset.")

No-show analysis by department:


,Total_Appointments,No_Show_Count,No_Show_Percentage
Department,,,
Pediatrics,107,16,14.95
Dermatology,105,15,14.29
Orthopedics,92,12,13.04
General,98,9,9.18
Cardiology,98,8,8.16



Department with the highest no-show percentage: Pediatrics


## 9. Compare SMS reminders with attendance status
This uses a cross-tabulation and row percentages. It compares the recorded attendance categories for SMS Reminder = Yes and No.

In [32]:
reminder_counts = pd.crosstab(data["SMS_Reminder"], data["Attendance_Status"])
reminder_percentages = pd.crosstab(
    data["SMS_Reminder"],
    data["Attendance_Status"],
    normalize="index"
) * 100

print("Attendance counts by SMS reminder:")
display(reminder_counts)

print("\nAttendance percentages within each reminder group:")
display(reminder_percentages.round(2))

Attendance counts by SMS reminder:


Attendance_Status,Attended,Cancelled,No-show
SMS_Reminder,,,
No,198,21,34
Yes,199,22,26



Attendance percentages within each reminder group:


Attendance_Status,Attended,Cancelled,No-show
SMS_Reminder,,,
No,78.26,8.30,13.44
Yes,80.57,8.91,10.53


## 10. Average waiting time: new vs follow-up appointments

In [33]:
appointment_wait_summary = data.groupby("Appointment_Type")["Wait_Time_Min"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print("Waiting-time comparison by appointment type:")
display(appointment_wait_summary)

Waiting-time comparison by appointment type:


,count,mean,median,min,max
Appointment_Type,,,,,
Follow-up,243,28.25,27.0,2.0,71.0
New,257,28.06,28.0,2.0,69.0


## 11. NumPy calculations: median, quartiles and standard deviation

In [34]:
wait_array = data["Wait_Time_Min"].to_numpy(dtype=float)

wait_median = np.median(wait_array)
wait_q1 = np.percentile(wait_array, 25)
wait_q3 = np.percentile(wait_array, 75)
wait_std = np.std(wait_array, ddof=0)

print(f"Median waiting time: {wait_median:.2f} minutes")
print(f"First quartile (Q1): {wait_q1:.2f} minutes")
print(f"Third quartile (Q3): {wait_q3:.2f} minutes")
print(f"Standard deviation: {wait_std:.2f} minutes")

Median waiting time: 28.00 minutes
First quartile (Q1): 18.00 minutes
Third quartile (Q3): 38.00 minutes
Standard deviation: 14.27 minutes


## 12. Identify unusually long waiting times using IQR
The Interquartile Range (IQR) is `Q3 - Q1`. An appointment is flagged as unusually long when its waiting time is above `Q3 + 1.5 * IQR`.
This is a statistical flag for operational review, not a medical judgment.

In [35]:
iqr = wait_q3 - wait_q1
upper_limit = wait_q3 + 1.5 * iqr

long_wait_records = data[data["Wait_Time_Min"] > upper_limit].copy()

print(f"IQR: {iqr:.2f} minutes")
print(f"Upper IQR limit: {upper_limit:.2f} minutes")
print("Number of unusually long waits:", len(long_wait_records))
print("\nFlagged records (first 10):")
display(long_wait_records[[
    "Appointment_ID", "Department", "Appointment_Date",
    "Wait_Time_Min", "Appointment_Type"
]].head(10))

IQR: 20.00 minutes
Upper IQR limit: 68.00 minutes
Number of unusually long waits: 2

Flagged records (first 10):


,Appointment_ID,Department,Appointment_Date,Wait_Time_Min,Appointment_Type
121,AP00121,Orthopedics,2025-05-17,69.0,New
226,AP00226,Orthopedics,2025-03-27,71.0,Follow-up


## 13. Appointment volume by date and department

In [36]:
date_department_volume = data.groupby(
    [data["Appointment_Date"].dt.date, "Department"]
)["Appointment_ID"].count().unstack(fill_value=0)

print("Appointment volume by date and department (first 10 dates):")
display(date_department_volume.head(10))

print("\nTotal appointments by month:")
print(data.groupby("Month")["Appointment_ID"].count().to_string())

Appointment volume by date and department (first 10 dates):


Department,Cardiology,Dermatology,General,Orthopedics,Pediatrics
Appointment_Date,,,,,
2025-01-01,2,0,0,0,1
2025-01-02,1,0,1,0,0
2025-01-03,1,2,1,0,0
2025-01-04,0,2,1,0,0
2025-01-05,0,0,0,0,1
2025-01-06,0,1,0,0,0
2025-01-07,1,0,2,0,0
2025-01-08,0,1,1,0,1
2025-01-09,0,0,1,1,0



Total appointments by month:
Month
2025-01     71
2025-02    106
2025-03     80
2025-04     82
2025-05     88
2025-06     73


## 14. Estimated consultation fees for attended appointments

In [37]:
attended = data[data["Attendance_Status"].astype(str).str.strip().str.lower() == "attended"].copy()

total_attended_fees = attended["Consultation_Fee"].sum()
fee_summary_by_department = attended.groupby("Department").agg(
    Attended_Visits=("Appointment_ID", "count"),
    Average_Fee=("Consultation_Fee", "mean"),
    Estimated_Total_Fee=("Consultation_Fee", "sum")
).round(2)

print("Attended appointments:", len(attended))
print("Estimated total consultation fees for attended appointments: ₹{:,.2f}".format(total_attended_fees))
print("\nEstimated fee summary by department:")
display(fee_summary_by_department)

Attended appointments: 397
Estimated total consultation fees for attended appointments: ₹296,200.00

Estimated fee summary by department:


,Attended_Visits,Average_Fee,Estimated_Total_Fee
Department,,,
Cardiology,82,712.20,58400
Dermatology,85,763.53,64900
General,78,796.15,62100
Orthopedics,71,680.28,48300
Pediatrics,81,771.60,62500


## 15. Lists, dictionaries, tuples and sets
This section demonstrates the Python collection types from the syllabus using the dataset's department and attendance values.

In [38]:
department_list = data["Department"].tolist()
department_set = set(department_list)
attendance_tuple = tuple(data["Attendance_Status"].dropna().unique())
department_counts_dict = data["Department"].value_counts().to_dict()

print("Unique departments (set):", department_set)
print("Attendance categories (tuple):", attendance_tuple)
print("Department appointment counts (dictionary):", department_counts_dict)

Unique departments (set): {'General', 'Orthopedics', 'Cardiology', 'Dermatology', 'Pediatrics'}
Attendance categories (tuple): ('Attended', 'No-show', 'Cancelled')
Department appointment counts (dictionary): {'Pediatrics': 107, 'Dermatology': 105, 'General': 98, 'Cardiology': 98, 'Orthopedics': 92}


## 16. Functions and a small OOP example
A function makes a calculation reusable. The class groups related summary operations into one object.

In [39]:
def calculate_percentage(part, whole):
    if whole == 0:
        return 0
    return (part / whole) * 100

class HospitalOperationsAnalysis:
    def __init__(self, appointment_data):
        self.data = appointment_data

    def total_appointments(self):
        return len(self.data)

    def average_waiting_time(self):
        return self.data["Wait_Time_Min"].mean()

    def attendance_summary(self):
        counts = self.data["Attendance_Status"].value_counts()
        percentages = counts.apply(lambda count: calculate_percentage(count, len(self.data)))
        return pd.DataFrame({"Count": counts, "Percentage": percentages.round(2)})

analysis = HospitalOperationsAnalysis(data)

print("Total appointments:", analysis.total_appointments())
print("Average waiting time: {:.2f} minutes".format(analysis.average_waiting_time()))
display(analysis.attendance_summary())

Total appointments: 500
Average waiting time: 28.15 minutes


,Count,Percentage
Attendance_Status,,
Attended,397,79.4
No-show,60,12.0
Cancelled,43,8.6


## 17. Operational recommendations
The following recommendation helper uses the calculated results. Review the printed values before submitting; recommendations are operational suggestions only.

In [40]:
recommendations = []

if len(no_show_rows) > 0:
    highest_no_show_department = department_no_show["No_Show_Percentage"].idxmax()
    highest_no_show_rate = department_no_show["No_Show_Percentage"].max()
    recommendations.append(
        f"Review appointment confirmation and reminder processes in {highest_no_show_department}, "
        f"which has the highest recorded no-show rate ({highest_no_show_rate:.2f}%)."
    )
else:
    recommendations.append(
        "The dataset has no records labelled 'No-show'; standardize attendance labels before drawing no-show conclusions."
    )

busiest_department = department_summary["Total_Appointments"].idxmax()
recommendations.append(
    f"Plan staff and appointment slots around {busiest_department}, the department with the highest appointment volume."
)

recommendations.append(
    f"Use the median waiting time ({wait_median:.2f} minutes) as a typical waiting-time reference, "
    "and monitor departments whose averages are higher."
)

recommendations.append(
    f"Review the {len(long_wait_records)} appointments flagged above the IQR upper limit "
    f"({upper_limit:.2f} minutes) to identify possible scheduling bottlenecks."
)

recommendations.append(
    "Compare attendance percentages for SMS Reminder = Yes and No before changing reminder policy; "
    "this dataset shows association, not proof that reminders cause attendance."
)

for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")

1. Review appointment confirmation and reminder processes in Pediatrics, which has the highest recorded no-show rate (14.95%).
2. Plan staff and appointment slots around Pediatrics, the department with the highest appointment volume.
3. Use the median waiting time (28.00 minutes) as a typical waiting-time reference, and monitor departments whose averages are higher.
4. Review the 2 appointments flagged above the IQR upper limit (68.00 minutes) to identify possible scheduling bottlenecks.
5. Compare attendance percentages for SMS Reminder = Yes and No before changing reminder policy; this dataset shows association, not proof that reminders cause attendance.


## 18. Basic testing and debugging checks
These checks help detect common mistakes such as negative waiting times, invalid fees, and duplicate appointment IDs.

In [41]:
# Test 1: Appointment IDs should be unique after duplicate removal.
assert data["Appointment_ID"].is_unique, "Duplicate appointment IDs remain."

# Test 2: Waiting time should not be negative.
assert (data["Wait_Time_Min"] >= 0).all(), "Negative waiting times found."

# Test 3: Consultation fees should not be negative.
assert (data["Consultation_Fee"] >= 0).all(), "Negative consultation fees found."

# Test 4: Attendance percentages should total approximately 100%.
assert abs(attendance_percentages.sum() - 100) < 0.01, "Attendance percentages do not total 100%."

print("All basic validation tests passed.")

All basic validation tests passed.


## 19. Save cleaned data and summary files
The CSV files are saved in the Colab session. Download them from the Files panel or use `files.download`.

In [42]:
data.to_csv("cleaned_hospital_appointments.csv", index=False)
department_summary.to_csv("department_summary.csv")
fee_summary_by_department.to_csv("attended_fee_summary.csv")

print("Saved files:")
print("- cleaned_hospital_appointments.csv")
print("- department_summary.csv")
print("- attended_fee_summary.csv")

# Uncomment these lines if you want to download the files directly:
# files.download("cleaned_hospital_appointments.csv")
# files.download("department_summary.csv")
# files.download("attended_fee_summary.csv")

Saved files:
- cleaned_hospital_appointments.csv
- department_summary.csv
- attended_fee_summary.csv


## 20. Conclusion
This project summarizes appointment operations using the supplied dataset. It calculates attendance percentages, department workload, waiting-time statistics, SMS reminder comparisons, no-show rates where such records exist, and estimated consultation fees for attended appointments. No graphs are used. Results should be interpreted as operational information only and not as medical conclusions.